# 01 Qwen 下载、缓存、加载和一次推理到底发生什么

先解决最底层的问题：Qwen 到底从哪里来？代码里哪几行负责下载/加载？一次回答又是怎么生成的？

本项目的模型名是：`Qwen/Qwen2.5-0.5B-Instruct`。Hugging Face 的 `from_pretrained` 会负责从本地缓存或网络加载模型。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("学习原则: 本 notebook 默认只读项目文件；所有真实推理/训练开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=None):
    lines = read_text(rel).splitlines()
    if end is None:
        end = len(lines)
    print(f"--- {rel}:{start}-{end} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if keyword in line:
            hits.append(idx)
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        start = max(1, hit - context)
        end = min(len(lines), hit + context)
        show_file(rel, start, end)
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 1. 模型缓存在哪里？

项目建议设置：

```powershell
$env:HF_HOME=(Resolve-Path -LiteralPath ".hf_cache").Path
```

这会让 Hugging Face 把模型和数据缓存放在项目内的 `.hf_cache`，避免缓存散落到系统目录。

In [ ]:
cache = PROJECT_ROOT / ".hf_cache"
print("HF cache exists:", cache.exists())
if cache.exists():
    for p in [cache / "hub", cache / "datasets"]:
        print(p.relative_to(PROJECT_ROOT), "exists:", p.exists())

## 2. infer.py 里哪里写了模型名？

In [ ]:
find_lines("scripts/infer.py", "DEFAULT_MODEL", context=4)

## 3. tokenizer 是怎么加载的？

tokenizer 负责把文字切成 token id。没有 tokenizer，模型看不懂文字。

In [ ]:
find_lines("scripts/infer.py", "AutoTokenizer.from_pretrained", context=6)

这几行的意思：

- `args.model_name` 是模型名，默认 Qwen2.5-0.5B-Instruct。
- `local_files_only=True` 时，只从本地缓存找，不联网下载。
- 如果 tokenizer 没有 pad token，就把 eos token 当 pad token，避免 batch padding 报错。

## 4. Qwen 模型本体是怎么加载的？

In [ ]:
find_lines("scripts/infer.py", "AutoModelForCausalLM.from_pretrained", context=10)

关键点：

- `AutoModelForCausalLM` 表示因果语言模型，也就是一个 token 接一个 token 地生成。
- `dtype` 决定 fp16/bf16/fp32，影响显存和速度。
- 有 CUDA 时用 `device_map="auto"`，让 transformers 自动放到 GPU。
- `local_files_only` 控制是否只读本地缓存。

## 5. 一次聊天 prompt 是怎么进入模型的？

In [ ]:
find_lines("scripts/infer.py", "apply_chat_template", context=12)

`apply_chat_template` 很关键。它把 system/user 消息变成 Qwen 训练时熟悉的聊天格式，再转成 token id。不同模型的聊天模板不一样，所以不要手写拼接，优先用 tokenizer 自带模板。

## 6. 模型如何生成答案？

In [ ]:
find_lines("scripts/infer.py", "model.generate", context=12)

生成流程：

1. 把 prompt token 放进模型。
2. `model.generate` 一个 token 一个 token 往后续。
3. 如果 `temperature=0`，基本是确定性输出，更适合做对比实验。
4. 最后只解码新生成的 tokens，不把原 prompt 打出来。

## 7. 安全练习：默认不真正跑模型

确认环境后可以把 `RUN_INFERENCE` 改成 True。它只推理，不训练，不写文件。

In [ ]:
prompt = "请用三点解释机器学习里的 LoRA 微调，不要解释成无线通信 LoRa。"
cmd = [sys.executable, "scripts/infer.py", "--prompt", prompt, "--max_new_tokens", "128", "--temperature", "0", "--local_files_only"]
print(" ".join(cmd))
RUN_INFERENCE = False
if RUN_INFERENCE:
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, encoding="utf-8", errors="replace")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("默认不运行。学习时先读代码，想实际推理再改 True。")